In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

### Install Cellpose-SAM

https://pypi.org/project/cellpose/

https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
# !pip3 install scikit-learn
# !pip3 install seaborn
# !pip3 install plotly

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(1, '../src/')

from Basic import *
from image_lib import *
from neural_network_lib import *

pd.set_option("display.precision", 3)
from IPython.display import display, HTML
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

#### Check if colab notebook instance has GPU access

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

In [ ]:
reload_np = False

if reload_np:
    del(np)
    import numpy as np
np.__version__

In [ ]:
# !pip3 install scispacy==0.6.2 --force-reinstall

In [ ]:
# !pip install "numpy<2.0" --force-reinstall
# !pip3 install opencv-python==4.9.0.80
# !pip3 install grad-cam

In [ ]:
root0 = "../../colaboracoes/deOcesano"
os.listdir(root0)

In [ ]:
cp = Cellpose(root0=root0, verbose=True)

```
dic_plate = {}
dic_plate['Plate1847'] = {}
dic_plate['Plate1847']['class_names'] = ['10pct', '1pct', '1pct_il1b']
dic_plate['Plate1847']['dir_names'] = ['10perc', '1perc', 'IL1B']
dic_plate['Plate1847']['probes'] = ['ATP5A1', 'P21']
dic_plate['Plate1847']['dir_origins'] = ['%s - 10%% SFB', '%s - 1%% SFB', '%s - 1%% SFB and IL-1B']


dic_plate['Plate1896'] = {}
dic_plate['Plate1896']['class_names'] = ['1pct', '1pct_il1b']
dic_plate['Plate1896']['dir_names'] = ['1perc', 'IL1B']
dic_plate['Plate1896']['probes'] = ['FACT', 'Faloidina', 'Rac1']
dic_plate['Plate1896']['dir_origins'] = ['%s - 1%% SFB', '%s - 1%% SFB and IL-1B']
```

In [ ]:
# !pip3 install pyyaml
cp.set_default_parameters(root_yaml=os.getcwd(), verbose=True)
# cp.dic_yml

### Root where the images (samples) are

In [ ]:
cp.root_samples

### Root where the crop images (ncropxncrop) are

In [ ]:
cp.root_crop

In [ ]:
plates = cp.list_plates(s_start='Plate')
plates

In [ ]:
i=0
plate=plates[i]
iprobe=0

cp.set_plate_params(plate=plate, verbose=True)

In [ ]:
probes = cp.probes
probes

### Experiments or Samples or Images

In [ ]:
exps = cp.list_experiments(flg_is_dir=True)
exps

In [ ]:
j=2
experiment = exps[j]
cp.create_roots_experiment(experiment, verbose=True)

# print(f"\n\nexperiment: '{cp.experiment}'  probe='{cp.probe}' {'is ok' if cp.probe in cp.probes else 'error'}")

### Images available

In [ ]:
fnames = cp.list_images_already_set(image_type='tif', verbose=True)
len(fnames), fnames[0]

### Get one image

In [ ]:
i=0
fname=fnames[i]
img = cp.read_PIL_image(fname, cp.root_image, verbose=True)
(width, height) = img.size
print(width, height)

In [ ]:
cp.display_img(img, figsize=(5,5));

### Crop dirs

In [ ]:
cp.root_crop, cp.root_image

In [ ]:
root_data, dic_root_train, dic_root_test = \
cp.set_data_origin_and_create_roots_to_train_and_test(image_example=img,verbose=True)

In [ ]:
print(cp.width, cp.height)
print(cp.size_x, cp.size_y)

In [ ]:
print(cp.model_fname)
print(cp.model_table)

In [ ]:
cp.clean_train_and_test(verbose=True)

### Sampling

In [ ]:
root_train_case = os.path.join(cp.root_train)
root_test_case = os.path.join(cp.root_test)

root_train_case, root_test_case

In [ ]:
cp.class_names, cp.dir_origins, cp.dir_names

In [ ]:
cp.root_data

In [ ]:
verbose=False

ret, df = cp.copy_data_train_test(max_images=1200, perc_train=.60, type_image='png', verbose=verbose)
ret

In [ ]:
df

In [ ]:
print(df.iloc[0].root_data)
print(df.iloc[0].root_target_train)
print(df.iloc[0].root_target_test)

### My Neural Network

In [ ]:
mnn = MyNN(root0=root0, verbose=True)

In [ ]:
train_transform, test_transform = mnn.create_data_transforms(cp.size_x, cp.size_y)

train_transform, test_transform

In [ ]:
cp.model_name, cp.plate, cp.experiment, cp.ncrop, cp.probe

In [ ]:
cp.root_crop

In [ ]:
df

In [ ]:
df.iloc[0].train_samples

# Prepare DataLoader

In [ ]:
train_dataset = ImageDataset(cp=cp, df=df, transform=train_transform, train_or_test='train')

In [ ]:
test_dataset = ImageDataset(cp=cp, df=df, transform=test_transform, train_or_test='test')

In [ ]:
train_dataset.samples[0], len(train_dataset.samples)

In [ ]:
test_dataset.samples[0], len(test_dataset.samples)

In [ ]:
batch_size = 10

dl_train = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dl_test  = torch.utils.data.DataLoader(test_dataset,  batch_size=batch_size, shuffle=True)

print('Number of training batches', len(dl_train))
print('Number of test batches', len(dl_test))

In [ ]:
for x in dl_train:
    print(x[0].shape, x[1])
    break

In [ ]:
images, labels = next(iter(dl_train))
print(images.shape)
print(labels.shape)

# Data Visualization

In [ ]:
class_names = train_dataset.class_names

# Creating the Model

In [ ]:
train_dataset.class_names

In [ ]:
train_dataset.class_to_idx

### Parameters

In [ ]:
lr=1e-5
n_classes = len(class_names)

### Model: ResNet18

In [ ]:
x[0].shape, size_x, size_y

In [ ]:
lr=1e-4
weight_decay=lr/2
n_classes = len(class_names)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(in_features=model.fc.in_features, out_features= n_classes)

model.fc.in_features, model.fc.out_features

In [ ]:
try:
    device = "cuda"
    model = model.to(device)
    print("Ok CUDA")
except:
    print("Error: device must be CUDA - however, it is not available")

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

from torch.optim.lr_scheduler import StepLR
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
def show_preds(model, fontsize=14, figsize=(12, 20)):
    model.eval()
    images, labels = next(iter(dl_test))
    images = images.to(device)
    
    print(images.shape, images[0].shape, labels.shape, labels[0])
    outputs = model(images)
    _, preds = torch.max(outputs, 1)
    # print(preds)
    show_images(images.cpu(), labels, preds.cpu(), fontsize=fontsize, figsize=figsize)

In [ ]:
show_preds(model, fontsize=12, figsize=(5, 10))

# Training the Model

In [ ]:
cp.root_data

In [ ]:
root_best = os.path.join(cp.root_data, 'best_model')
create_dir(root_best)


fname_best_model= os.path.join(root_best, fname0)

fname_best_model

In [ ]:
import time

### Eval accuracy

In [ ]:
def evaluate_loss_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = nn.CrossEntropyLoss()(outputs, labels)
                preds = outputs.argmax(1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return loss_sum / len(loader), correct / total    

### Treinamento com Mixed Precision (AMP) — 2× mais rápido

#### Early Stopping + Best Weights / training
#### Avoid to train to much and worsing the accuracy

In [ ]:
scaler = torch.amp.GradScaler('cuda')

best_acc = 0

In [ ]:
train_losses = []
val_losses = []
val_accuracies = []

In [ ]:
epochs=100
early_stop_patience=20

epochs=500
early_stop_patience=50

wait = 0

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    start = time.time()

    for images, labels in dl_train:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = loss_fn(outputs, labels) # cross-entropy

        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    # acc = eval_accuracy(model, dl_test, device)
    train_loss = epoch_loss / len(dl_train)
    val_loss, val_acc = evaluate_loss_accuracy(model, dl_test)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    elapsed = time.time() - start
    print(f"{epoch+1}/{len(train_losses)}) loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, LR: {lr:.2e} Time: {elapsed:.1f}s")

    if val_acc > best_acc:
        best_acc = val_acc
        wait = 0
        try:
            torch.save(model.state_dict(), fname_best_model)
            print(f"Model saved at: '{fname_best_model}'")
        except:
            print(f"Error: model not saved at: '{fname_best_model}'")
    else:
        wait += 1
        if wait >= early_stop_patience:
            print("Early stopping! Next epoch ...")
            break
        print(">>> wait", wait)

print(f"\nTraining finished. Best accuracy = {best_acc:.4f}")
        

In [ ]:
len(val_losses), len(train_losses)

In [ ]:
fname_best_model

In [ ]:
dfacc = pd.DataFrame({'train_loss': train_losses, 'val_loss': val_losses, 'accuracy': val_accuracies})

fname=fname.replace('.pt','.tsv')
filename= os.path.join(root_best, fname0)

fname_best_model

dfacc

In [ ]:
epochs_range = range(1, len(train_losses) + 1)
plt.figure(figsize=(12,5))

#------ loss plot ------------------
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, marker='o', label='Train Loss')
plt.plot(epochs_range, val_losses, marker='o', label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)

#----- accuracy plot ---------------
plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_accuracies, marker='o', label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Recover best model

In [ ]:
new_model = model
new_model.load_state_dict(torch.load(fname_best_model, map_location=device))
# new_model.eval()

In [ ]:
show_preds(new_model, fontsize=12, figsize=(5, 10))